In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat
import pickle

from utils_clique import (
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    plot_cliques,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)
from scipy.io import loadmat

probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
recording_raw = se.read_intan(f"/home/ubuntu/Documents/jct/project/251205/M190011_260121_150111_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)

recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

rec_params_raw = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
rec_params_raw = rec_params_raw[rec_params_raw['bhv_codes'] == 10]

original_fs = 30000
target_fs = 10000
fs_ratio = original_fs / target_fs
rec_params_raw['rec_codes_points_10000'] = (rec_params_raw['rec_codes_points'] / fs_ratio).astype(int)
rec_params_raw = rec_params_raw[(rec_params_raw['trial_ids'] >= 300) & (rec_params_raw['trial_ids'] < 5300)]
start_sample = rec_params_raw['rec_codes_points_10000'].iloc[0]
end_sample = rec_params_raw['rec_codes_points_10000'].iloc[-1]

recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)
recording_segment = recording_segment.save(format="binary", n_jobs = 30)   


read success
Use cache_folder=/tmp/spikeinterface_cache/tmpk1nufrj8/0WERFHWI
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=4.88 MiB - total_memory=146.48 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 3726/3726 [03:49<00:00, 16.25it/s]


In [3]:
output_folder = '/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/'
probe.set_contact_ids(recording_segment.channel_ids)

cliques = build_sliding_cliques(
    probe,
    clique_size=32,
    min_size=25,
    min_overlap=6,
    target_groups=10,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 32,
        'min_size': 25,
        'min_overlap': 6,
        'target_groups': 10,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

plot_cliques(probe, cliques, f'{output_folder}/cliques.pdf')

[INFO] Built 10 cliques (target 10)
       Clique 00: channels 204-39 (32 channels)
       Clique 01: channels 74-81 (32 channels)
       Clique 02: channels 174-227 (32 channels)
       Clique 03: channels 33-88 (32 channels)
       Clique 04: channels 69-253 (32 channels)
       Clique 05: channels 229-232 (32 channels)
       Clique 06: channels 35-24 (32 channels)
       Clique 07: channels 207-141 (32 channels)
       Clique 08: channels 134-109 (32 channels)
       Clique 09: channels 148-115 (32 channels)

Clique可视化PDF已保存至: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort//cliques.pdf


In [4]:
output_folder = '/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/'

for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"训练 Clique {clique_id}")
    print(f"{'='*60}")
    
    neuron_inf_path =  f'/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_{clique_id}/neuron_inf_all.pickle'

    gt_detect_array_path = f'/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_{clique_id}/gt_detect_array_all.csv'
    
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_dict = pickle.load(f)
    gt_detect_array = pd.read_csv(gt_detect_array_path)
    
    clique_channel_ids = set(str(ch_id) for ch_id in clique.contact_ids)

    neuron_inf_segment = neuron_inf_dict_to_dataframe(neuron_inf_dict)
    recording_clique = get_recording_clique(recording_segment, clique)
    print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")

    recording_channel_ids = recording_clique.get_channel_ids()
    probe_to_clique_index = {str(ch_id): idx for idx, ch_id in enumerate(recording_channel_ids)}

    detect_channel = {
        1: [],  # 正极性通道列表
        -1: []  # 负极性通道列表
    }

    for _, neuron in neuron_inf_segment.iterrows():
        extremum_channel = neuron['extremum_channel']
        sign = neuron.get('sign', -1)  # 默认负极性
        if str(extremum_channel) in probe_to_clique_index:
            clique_idx = probe_to_clique_index[str(extremum_channel)]
            detect_channel[sign].append(clique_idx)

    detect_channel[1] = set(detect_channel[1])
    detect_channel[-1] = set(detect_channel[-1])

    print(f"  正极性通道数: {len(detect_channel[1])}")
    print(f"  负极性通道数: {len(detect_channel[-1])}")

    gt_detect_array['extremum_channel'] = gt_detect_array['extremum_channel'].astype(str)
    clique_save_dir = f'{output_folder}/clique_{clique_id}/'
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,
        gt_detect_array=gt_detect_array,
        neuron_inf=neuron_inf_segment,
        save_dir=clique_save_dir,
        duration_seconds=1500,
        thr_min=5,
        thr_max=35,
        distance=3,
        wlen=5,
        prominence=15,
        left_sample=10,
        right_sample=20,
        max_firing_channel=None,
        detect_channel=detect_channel,
        chunk_number=10,  
        n_jobs=30  
    )
    
    n_channels = recording_clique.get_num_channels()
    n_repeats = 1
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
        
        #print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
    
    print(f"  Clique {clique_id} 所有重复训练完成!")

print("\n所有训练完成！")



训练 Clique 0
  Recording clique channels: 32
  正极性通道数: 8
  负极性通道数: 26
### 1. Threshold Detection and Waveform Extraction
Sampling rate: 10000.0 Hz, Number of channels: 32
Recording total length: 37255467 samples (3725.55 seconds)
Will process first 15000000 samples (1500.00 seconds)
Chunk number for memory efficiency: 10
Chunk size: 1500000 samples (150.00 seconds)
gt_detect_array_filtered: 91450


Chunk full pipeline: 100%|██████████| 10/10 [00:30<00:00,  3.06s/chunk]



GT匹配统计(整合): 87277/91450 GT spikes被检测到 (召回率: 0.9544)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_0/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_0/train_data
Data statistics:
  - Total spike count: 194191
  - Number of channels: 30
  - Window length: 32
  - Number of unique units: 46
  - Noise spike count: 105120
  - Valid spike count: 89071
  - Unmatched GT spikes added: 4172

  ===== 重复训练 1/1 =====
Using device: cuda
Create dataset...
Auto-extracting keep_id from data
Dataset loaded:
  - Total samples: 194191
  - Number of channels: 30
  - Window length: 32
  - Number of unique units: 46
  - Noise samples: 105120.0
  - Non-noise samples: 89071.0
Model parameters:
  - Number of chann

Chunk full pipeline: 100%|██████████| 10/10 [00:32<00:00,  3.26s/chunk]



GT匹配统计(整合): 132131/135091 GT spikes被检测到 (召回率: 0.9781)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_1/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_1/train_data
Data statistics:
  - Total spike count: 227813
  - Number of channels: 30
  - Window length: 32
  - Number of unique units: 42
  - Noise spike count: 95013
  - Valid spike count: 132800
  - Unmatched GT spikes added: 2960

  ===== 重复训练 1/1 =====
Using device: cuda
Create dataset...
Auto-extracting keep_id from data
Dataset loaded:
  - Total samples: 227813
  - Number of channels: 30
  - Window length: 32
  - Number of unique units: 42
  - Noise samples: 95013.0
  - Non-noise samples: 132800.0
Model parameters:
  - Number of cha

...

epoch : 16/20, val detection loss = 65.956610, classification loss = 1141.233247
epoch : 16/20, val acc noise = 0.8675, val acc label = 0.9585
epoch : 17/20


Training: 100%|██████████| 85/85 [00:00<00:00, 140.79it/s]


epoch : 17/20, detection loss = 1.502958, classification loss = 395.603730


Validation: 100%|██████████| 22/22 [00:00<00:00, 204.21it/s]

epoch : 17/20, val detection loss = 57.148346, classification loss = 1159.233398
epoch : 17/20, val acc noise = 0.8705, val acc label = 0.9639
Early stopping triggered at epoch 17
Best model was at epoch 12 with val_loss = 1180.134452

Dataset split:
  - Training set: 43412 samples
  - Validation set: 10853 samples
Final model saved
Training log saved to: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort//clique_9//model_1/training_log.csv
  Clique 9 所有重复训练完成!

所有训练完成！
